# Training Gru with 1 layer  

## 1.1 import all library 

In [1]:
#data manipulation library
import pandas as pd
import numpy as np

import os 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow logging
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use only the first GPU
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # Enable oneDNN optimizations


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import random
import time 

import tensorflow as tf
import sklearn

from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import plot_model
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from datetime import timedelta
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from matplotlib import pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

print("all imports successful")


TensorFlow version: 2.21.0
Scikit-learn version: 1.7.2
Num GPUs Available: 0
all imports successful


## 1.2 DATA LOADED

In [2]:
DATA = {

    "train": "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/train_data_class_weight_80_new.xlsx", #80 % of train data
    "test" : "/home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/test_data_class_weight_20_new.xlsx" #20 percent of train data
}

print("data paths defined successfully")

def load_data(file_path,name):

    df = pd.read_excel(file_path)
    
    X = df.drop(columns=["label"]).values
    y = df["label"].values

    print(f"{name.upper()} DATA")
    print(f"Data loaded from {file_path} successfully. Shape: {X.shape}, {y.shape}")

    label_dist = pd.Series(y).value_counts().rename(index={0: "Label 0", 1: "Label 1"})
    print("Label distribution:\n", label_dist, "\n")

    return X, y

X_train, y_train = load_data(DATA["train"], "train")
X_test, y_test   = load_data(DATA["test"], "test")

data paths defined successfully
TRAIN DATA
Data loaded from /home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/train_data_class_weight_80_new.xlsx successfully. Shape: (9532, 253), (9532,)
Label distribution:
 Label 1    4769
Label 0    4763
Name: count, dtype: int64 

TEST DATA
Data loaded from /home/gibannn/kuliah/sem3/paper/SMILES2VEC/data/AHR/test_data_class_weight_20_new.xlsx successfully. Shape: (1344, 253), (1344,)
Label distribution:
 Label 0    1192
Label 1     152
Name: count, dtype: int64 



## 1.3 define vocab size and max length 


In [3]:
#define vocab size and max length of sequence
vocab_size = int (
    max(X_train.max(), 
        X_test.max()) + 1
    )

max_length = int(
    max(
        X_train.shape[1], 
        X_test.shape[1], 
        )
)

print(f"Vocabulary size: {vocab_size}")
print(f"Maximum sequence length: {max_length}")

#define class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_test),
    y=y_train
)

weights = {i: weight for i, weight in enumerate(class_weights)}
print("Class weights:", weights)

#define distribution in val, test, and train set
print("Train set distribution:\n", pd.Series(y_train).value_counts(normalize=True), "\n")
print("Test set distribution:\n", pd.Series(y_test).value_counts(normalize=True), "\n")

Vocabulary size: 112
Maximum sequence length: 253
Class weights: {0: np.float64(1.0006298551333193), 1: np.float64(0.9993709373034179)}
Train set distribution:
 1    0.500315
0    0.499685
Name: proportion, dtype: float64 

Test set distribution:
 0    0.886905
1    0.113095
Name: proportion, dtype: float64 



## 1.4 define seeds and set to gpu 

In [4]:
Random_seed = 116

tf.random.set_seed(Random_seed)
np.random.seed(Random_seed)
random.seed(Random_seed)

def set_gpu_memory_growth():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("Memory growth enabled for GPUs")
        except RuntimeError as e:
            print(f"Error setting memory growth: {e}")
    else:
        print("No GPUs found")

set_gpu_memory_growth()

learning_rate = 0.001
batch_size = 256
epochs = 50

print("training parameters have been set :")
print(f"Learning Rate: {learning_rate}")
print(f"Batch Size: {batch_size}")
print(f"Epochs: {epochs}")
print("")

No GPUs found
training parameters have been set :
Learning Rate: 0.001
Batch Size: 256
Epochs: 50



## 1.5 set directory to save the model,image model architecture,history,csv

In [5]:
Model = "./model_new"
image = "./image_new"
csv = "./csv_new"

os.makedirs(Model, exist_ok=True)
os.makedirs(image, exist_ok=True)
os.makedirs(csv, exist_ok=True)

Model_path = os.path.join(Model, "best_bigru_model.keras")
print(f"Model will be saved to: {Model_path}")

image_path = os.path.join(image, "training_history.png")
print(f"Training history plot will be saved to: {image_path}")

csv_path = os.path.join(csv, "training_history.csv")
print(f"Training history CSV will be saved to: {csv_path}")



Model will be saved to: ./model_new/best_bigru_model.keras
Training history plot will be saved to: ./image_new/training_history.png
Training history CSV will be saved to: ./csv_new/training_history.csv


## 1.6 Define the model architecture 1-layer Bigru 32/64 and confusiion matrix

In [6]:
#building the model
def build_model(vocab_size, max_length):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
        Bidirectional(GRU(64, return_sequences=False)),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])
    return model

#plotting the model architecture
def plot_model_architecture(model, image_path, show_shapes=True, show_layer_names=True):
    plot_model(
        model,
        to_file=image_path,
        show_shapes=show_shapes,
        show_layer_names=show_layer_names
    )

print("Building model...")
model = build_model(vocab_size, max_length)
print("Model built successfully")
print(f"The model diagram will be saved to: {image_path}")

Building model...
Model built successfully
The model diagram will be saved to: ./image_new/training_history.png


In [7]:
#compiling the model
def compile_model(model, learning_rate):
    
    model = model
    loss_function = tf.keras.losses.BinaryFocalCrossentropy(gamma=2, alpha=0.8, label_smoothing=0.1)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=loss_function,
        metrics=['accuracy']
    )

model.build(input_shape=(None, max_length))
print("model has been built successfully")



model has been built successfully


In [8]:
#defineing callbacks for training
def callbacks_model(
    Model_path,
    patience=5,
    monitor="val_loss",
    mode="min"
):
    early_stopping = EarlyStopping(
        monitor=monitor,
        patience=patience,
        restore_best_weights=True,
        mode=mode
    )
    
    model_checkpoint = ModelCheckpoint(
        filepath=Model_path,
        monitor=monitor,
        save_best_only=True,
        mode=mode
    )
    
    return [early_stopping, model_checkpoint]

print("Callbacks have been defined successfully. Model checkpoints will be saved to:", {Model_path})

Callbacks have been defined successfully. Model checkpoints will be saved to: {'./model_new/best_bigru_model.keras'}


In [9]:
#define the confusion matrix and classification report function
def compute_classification_metrics(y_true, y_pred):
    
    cm = confusion_matrix(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)

    metrics_dict = {
        "Confusion Matrix": cm,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy
    }
    
    return metrics_dict

print("Classification metrics function defined successfully")

Classification metrics function defined successfully


In [10]:
def classweight(y_train):
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    tuningweights = {0: 0.5, 1: 2.0}
    return tuningweights

print("Class weight function defined successfully")

Class weight function defined successfully


In [11]:
def dataframe_history(history):
    history_dict = {
        "Model": model.name,
        "Accuracy": metrics_dict["Accuracy"],
        "Precision": metrics_dict["Precision"],
        "Recall": metrics_dict["Recall"],
        "F1": metrics_dict["F1-Score"],
        "TP": metrics_dict["TP"],
        "TN": metrics_dict["TN"],
        "FP": metrics_dict["FP"],
        "FN": metrics_dict["FN"],
        "Training_Time": str(timedelta(seconds=int(training_time))),
        "Epochs": len(history.history["loss"])
    }
    df_history = pd.DataFrame(history_dict)
    return df_history

def create_df_history(history, metrics_dict, training_time):
    history_dict = {
        "Model": model.name,
        "Accuracy": metrics_dict["Accuracy"],
        "Precision": metrics_dict["Precision"],
        "Recall": metrics_dict["Recall"],
        "F1": metrics_dict["F1 Score"],
        "TP": metrics_dict["Confusion Matrix"][1][1],
        "TN": metrics_dict["Confusion Matrix"][0][0],
        "FP": metrics_dict["Confusion Matrix"][0][1],
        "FN": metrics_dict["Confusion Matrix"][1][0],
        "Training_Time": str(timedelta(seconds=int(training_time))),
        "Epochs": len(history.history["loss"])
    }
    df_history = pd.DataFrame([history_dict])
    return df_history
print("Dataframe history function defined successfully")

Dataframe history function defined successfully


In [12]:
def df_to_numpy(df):
    return df.values if hasattr(df, "values") else df.to_numpy()
print("Dataframe to numpy function defined successfully")

Dataframe to numpy function defined successfully


## 1.7 Define training function 

In [13]:
def train_model(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    model_name="BiGRU",
    model_save_path="./model/best_model.keras",
    patience=5,
    verbose=1,
    tuningweights=None
):

    print(f"Training {model_name} model")
    print("="*50)

    start_time = time.time()

    
    model = build_model(vocab_size, max_length)
    
    compile_model(model, learning_rate)

    callbacks = callbacks_model(model_save_path, patience=patience)

    
    history = model.fit(
        X_train, y_train,
        class_weight=tuningweights,
        callbacks=callbacks,
        batch_size=batch_size,
        epochs=epochs,
        verbose=verbose
    )
    training_time = time.time() - start_time
    return model, history ,training_time

model.summary()
print("Model summary printed successfully")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 253, 128)       │        14,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,961 (347.50 KB)

 Trainable params: 88,961 (347.50 KB)

 Non-trainable params: 0 (0.00 B)

Model summary printed successfully


In [14]:
def evaluate_model_datatest(model, X_test, y_test):
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    metrics_dict = compute_classification_metrics(y_test, y_pred)
    return metrics_dict

def evaluate_model_datatrain(model, X_train, y_train):
    y_pred_prob = model.predict(X_train)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    metrics_dict = compute_classification_metrics(y_train, y_pred)
    return metrics_dict

print("Evaluation functions defined successfully")

Evaluation functions defined successfully


In [15]:
weight_options = [
    {0: 1.0, 1: 5.0},      # Conservative
    {0: 1.0, 1: 7.0},      # Close to actual ratio
    {0: 1.0, 1: 7.8},      # Exact imbalance ratio (88.69% vs 11.31%)
    {0: 1.0, 1: 8.0},      # Match rounded ratio
    weights,                 # Original balanced weights
]

results_all = []
best_f1 = 0
best_model = None
best_history = None
best_weights = None

for i, w in enumerate(weight_options):
    print(f"\n{'='*80}")
    print(f"TRAINING WITH WEIGHT OPTION {i+1}/4: {w}")
    print('='*80)
    
    tuningweights = w
    
    model, history, training_time = train_model(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        model_name="BiGRU",
        model_save_path=Model_path,
        patience=1,
        verbose=1,
        tuningweights=tuningweights
    )
    
    print("finished training the model successfully")
    
    # Evaluate
    metrics_dict = evaluate_model_datatest(model, X_test, y_test)
    
    result_row = {
        "Weight_Option": str(i+1),
        "Weights": str(w),
        "Accuracy": metrics_dict["Accuracy"],
        "Precision": metrics_dict["Precision"],
        "Recall": metrics_dict["Recall"],
        "F1": metrics_dict["F1 Score"],
        "Epochs": len(history.history["loss"])
    }
    
    results_all.append(result_row)
    
    # Track best model by F1 score
    if metrics_dict["F1 Score"] > best_f1:
        best_f1 = metrics_dict["F1 Score"]
        best_model = model
        best_history = history
        best_weights = w
    
    print(f"Weight Option {i+1} - F1: {metrics_dict['F1 Score']:.4f}, Accuracy: {metrics_dict['Accuracy']:.4f}")

# Display comparison
print("\n" + "="*80)
print("WEIGHT COMPARISON RESULTS")
print("="*80)
comparison_df = pd.DataFrame(results_all)
print(comparison_df.to_string(index=False))

print(f"\nBest Model: Weight Option with F1 = {best_f1:.4f}")
print(f"Best Weights: {best_weights}")

# Save comparison table
comparison_df.to_csv(os.path.join(csv, "weight_comparison.csv"), index=False)
print(f"\nComparison saved to: {os.path.join(csv, 'weight_comparison.csv')}")


TRAINING WITH WEIGHT OPTION 1/4: {0: 1.0, 1: 5.0}
Training BiGRU model
Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 22s 463ms/step - accuracy: 0.5372 - loss: 0.3293
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 20s 514ms/step - accuracy: 0.8304 - loss: 0.1436
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 450ms/step - accuracy: 0.8569 - loss: 0.1272
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 22s 497ms/step - accuracy: 0.8644 - loss: 0.1220
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 456ms/step - accuracy: 0.8627 - loss: 0.1223
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 451ms/step - accuracy: 0.8634 - loss: 0.1198
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 19s 498ms/step - accuracy: 0.8654 - loss: 0.1175
Epoch 8/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 452ms/step - accuracy: 0.8651 - loss: 0.1165
Epoch 9/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 19s 499ms/step - accuracy: 0.8671 - loss: 0.1159
Epoch 10/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 440ms/step - accuracy: 0.8660 - loss: 0.1148
Epoch 11/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 459ms/step - a

## 1.8 evaluate the model on training and test data 

In [21]:
def evaluate_model_datatest(model, X_test, y_test):
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.55).astype(int).flatten()
    metrics_dict = compute_classification_metrics(y_test, y_pred)
    return metrics_dict

metrics_dict = evaluate_model_datatest(model, X_test, y_test)
print("Model evaluation on test data completed successfully")

def evaluate_model_datatrain(model, X_train, y_train):
    y_pred_prob = model.predict(X_train)
    y_pred = (y_pred_prob > 0.55).astype(int).flatten()
    metrics_dict = compute_classification_metrics(y_train, y_pred)
    return metrics_dict

metrics_dict_train = evaluate_model_datatrain(model, X_train, y_train)
print("Model evaluation on training data completed successfully")

def print_dataframe_csv(metrics_dict, filename):
    df_history = create_df_history(history, metrics_dict, training_time)
    df_history.to_csv(filename, index=False)
    print(f"Training history saved to {filename} successfully")

print_dataframe_csv(metrics_dict, csv_path)


42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step
Model evaluation on test data completed successfully
298/298 ━━━━━━━━━━━━━━━━━━━━ 19s 62ms/step
Model evaluation on training data completed successfully
Training history saved to ./csv_new/training_history.csv successfully


In [22]:
# Show test data results
results_df_test = create_df_history(history, metrics_dict, training_time)
print("\n" + "="*80)
print("TEST DATA RESULTS")
print("="*80)
print(results_df_test.to_string(index=False))

# Show training data results
results_df_train = create_df_history(history, metrics_dict_train, training_time)
print("\n" + "="*80)
print("TRAINING DATA RESULTS")
print("="*80)
print(results_df_train.to_string(index=False))


TEST DATA RESULTS
       Model  Accuracy  Precision   Recall       F1  TP  TN  FP  FN Training_Time  Epochs
sequential_1  0.809524   0.326667 0.644737 0.433628  98 990 202  54       0:14:57      50

TRAINING DATA RESULTS
       Model  Accuracy  Precision  Recall       F1   TP   TN  FP  FN Training_Time  Epochs
sequential_1  0.906525   0.858676 0.97337 0.912432 4642 3999 764 127       0:14:57      50


In [23]:
def evaluate_with_threshold(model, X_test, y_test, threshold=0.5):
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = (y_pred_prob > threshold).astype(int).flatten()
    metrics = compute_classification_metrics(y_test, y_pred)
    return metrics, threshold

# Try different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
results = []

for thresh in thresholds:
    metrics, t = evaluate_with_threshold(model, X_test, y_test, threshold=thresh)
    results.append({
        "Threshold": t,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1": metrics["F1 Score"]
    })

import pandas as pd
threshold_df = pd.DataFrame(results)
print(threshold_df.to_string(index=False))

 Threshold  Accuracy  Precision   Recall       F1
       0.3  0.476935   0.173964 0.967105 0.294885
       0.4  0.583333   0.205202 0.934211 0.336493
       0.5  0.734375   0.267574 0.776316 0.397976
       0.6  0.849702   0.373737 0.486842 0.422857
       0.7  0.889881   0.535714 0.197368 0.288462
       0.8  0.887649   0.555556 0.032895 0.062112


In [24]:
# Threshold analysis on best model
print("\n" + "="*80)
print("THRESHOLD TUNING FOR BEST MODEL")
print("="*80)

# Use best model from weight comparison
model = best_model
history = best_history

# Get predictions on test set
y_pred_prob = model.predict(X_test, verbose=1).flatten()

# Test different thresholds
thresholds = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55,0.555, 0.6, 0.65, 0.7]
threshold_results = []

for thresh in thresholds:
    y_pred = (y_pred_prob > thresh).astype(int)
    metrics = compute_classification_metrics(y_test, y_pred)
    
    threshold_results.append({
        "Threshold": f"{thresh:.2f}",
        "Accuracy": f"{metrics['Accuracy']:.4f}",
        "Precision": f"{metrics['Precision']:.4f}",
        "Recall": f"{metrics['Recall']:.4f}",
        "F1": f"{metrics['F1 Score']:.4f}"
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))

# Save threshold analysis
threshold_df.to_csv(os.path.join(csv, "threshold_analysis.csv"), index=False)
print(f"\nThreshold analysis saved to: {os.path.join(csv, 'threshold_analysis.csv')}")

# Find best threshold by F1
f1_scores = []
for thresh in thresholds:
    y_pred = (y_pred_prob > thresh).astype(int)
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal Threshold: {best_threshold:.2f} with F1 = {max(f1_scores):.4f}")


THRESHOLD TUNING FOR BEST MODEL
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step
Threshold Accuracy Precision Recall     F1
     0.30   0.4769    0.1740 0.9671 0.2949
     0.35   0.5283    0.1878 0.9539 0.3139
     0.40   0.5833    0.2052 0.9342 0.3365
     0.45   0.6510    0.2281 0.8750 0.3619
     0.50   0.7344    0.2676 0.7763 0.3980
     0.55   0.8095    0.3267 0.6447 0.4336
     0.56   0.8170    0.3357 0.6316 0.4384
     0.60   0.8497    0.3737 0.4868 0.4229
     0.65   0.8802    0.4622 0.3618 0.4059
     0.70   0.8899    0.5357 0.1974 0.2885

Threshold analysis saved to: ./csv_new/threshold_analysis.csv

Optimal Threshold: 0.56 with F1 = 0.4384


In [25]:
#cek class distribution in test set
print("\nTest set class distribution:")
value_counts = pd.Series(y_test).value_counts().sort_index()
total = len(y_test)

for label in [0, 1]:
    count = value_counts.get(label, 0)
    percentage = (count / total) * 100
    print(f"Label {label}: {count:,} samples out of {total} ({percentage:.2f}%)")


Test set class distribution:
Label 0: 1,192 samples out of 1344 (88.69%)
Label 1: 152 samples out of 1344 (11.31%)
